# Exercise 2 — MCPServer

An **MCPServer** hosts callable tools.  Use the `@server.tool()` decorator to register a function.  `list_tools()` returns the schema list for discovery; `call_tool(name, args)` invokes the function with keyword arguments from the args dict.  Errors never raise — they come back as error strings.

In [ ]:
import json
from dataclasses import dataclass, field
@dataclass
class MCPToolDef:
    name: str
    description: str
    input_schema: dict = field(default_factory=dict)

def tool_schema_text(tools):
    lines = []
    for t in tools:
        params = ", ".join(t.input_schema.keys())
        lines.append("- " + t.name + "(" + params + "): " + t.description)
    return "\n".join(lines)

# ── Exercise: implement MCPServer ────────────────────────────────────────────

class MCPServer:
    """An in-process MCP server that registers and calls tools."""

    def __init__(self, name="mcp_server"):
        self.name = name
        self._tools = {}

    def tool(self, name, description, schema=None):
        """Return a decorator that registers the function as a named MCP tool."""
        def _decorator(fn):
            # TODO: store {"def": MCPToolDef(name, description, schema or {}),
            #              "fn": fn} in self._tools[name]
            return fn
        return _decorator

    def list_tools(self):
        # TODO: return a list of entry["def"] for each entry in self._tools.values()
        return []

    def call_tool(self, name, args):
        # TODO: look up entry in self._tools; call entry["fn"](**args); return str(result)
        # If name is not in self._tools, return "Error: unknown tool ..."
        # Never raise — catch exceptions and return "Error: ..."
        return "Error: not implemented"


### Checks

In [ ]:
checks = 0

# 1 — MCPServer creates
try:
    server = MCPServer("test_server")
    assert server.name == "test_server"
    checks += 1; print("✅ 1 MCPServer created with name")
except Exception as e:
    print("❌ 1:", e)

# 2 — @server.tool decorator preserves the function
try:
    server = MCPServer("s")
    @server.tool("word_count", "Count words.", {"text": "text to count"})
    def word_count(text):
        return str(len(str(text).split()))
    assert word_count("hello world") == "2"
    checks += 1; print("✅ 2 @server.tool decorator preserves the function")
except Exception as e:
    print("❌ 2:", e)

# 3 — list_tools returns MCPToolDef list
try:
    server = MCPServer("s")
    @server.tool("wc", "Count.", {"text": "str"})
    def wc(text): return str(len(text.split()))
    tools = server.list_tools()
    assert len(tools) == 1 and tools[0].name == "wc"
    checks += 1; print("✅ 3 list_tools returns MCPToolDef list")
except Exception as e:
    print("❌ 3:", e)

# 4 — call_tool invokes the function with **args
try:
    server = MCPServer("s")
    @server.tool("upper", "Uppercase.", {"text": "str"})
    def upper(text): return str(text).upper()
    assert server.call_tool("upper", {"text": "hello"}) == "HELLO"
    checks += 1; print("✅ 4 call_tool returns correct result")
except Exception as e:
    print("❌ 4:", e)

# 5 — call_tool returns error string for unknown tool (does not raise)
try:
    server = MCPServer("s")
    result = server.call_tool("no_such_tool", {})
    assert isinstance(result, str)
    assert "error" in result.lower() or "unknown" in result.lower() or "no_such_tool" in result
    checks += 1; print("✅ 5 call_tool returns error string for unknown tool")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
